# MedEvidence Copilot: run Week 3 and Week 4

This notebook runs the real project code from the `aakash_r_changes` branch: the Week 3 agent and the Week 4 serving stack. The code, keys and certificates all come from your Google Drive, the same way as the Week 1 and 2 notebook.

| Part | What happens |
|---|---|
| **Setup** | Clone the repo, install packages, connect to the course endpoints, download and check the data, build the search index and the FAISS store |
| **Week 3** | Unit tests, the safety gate, one question traced step by step, a follow-up in the same session, the agent on a sample, the full evaluation, failures, traces, ROUGE-L |
| **Week 4** | Start the API, test caching, rate limiting and sessions, open the Streamlit UI, look at the Docker setup |
| **Finish** | Compare with the committed results and save everything to Google Drive |

## Before you run

1. **Runtime:** Runtime > Change runtime type > **T4 GPU**. The MedCPT reranker is much slower without it.
2. **Google Drive:** the folder `medevidence-copilot-colab` must contain:

   | File | What it is |
   |---|---|
   | `code/medevidence-copilot-week3-4.bundle` | The project code with the Week 3 and 4 commits, with its git history |
   | `secrets/medevidence.env` | Endpoint settings and API keys. Keep this folder private |
   | `certs/vm04-ca.crt`, `certs/vm02-ca.crt` | Endpoint certificates |

3. Allow Drive access when Colab asks.

## What it costs

Runtime > **Run all** uses the default settings below: everything except the two full agent runs.

| Step | Default | Model calls | Tokens | Time |
|---|---|---|---|---|
| Setup, data, index, FAISS | on | 1 test call | very few | about 5 min (about 15 min if the embeddings are not saved in Drive yet) |
| Unit tests | on | none | none | under 1 min |
| One question traced, plus a follow-up | on | about 10 | about 10K | about 1 min |
| Agent sample (10 questions) | on | about 45 | about 45K | about 3 min |
| API tests | on | about 5 | about 5K | about 2 min |
| **Full Week 3 run** (100 questions) | **off** | about 420 | **about 400K** | about 25 min |
| **Critic ablation** (100 questions, no critic) | **off** | about 300 | **about 300K** | about 20 min |

The full runs use your team's shared token quota, so they are off until you switch them on.
Every model answer is saved to Drive. If Colab disconnects, run the notebook again and it continues without paying for finished calls.

## Settings

In [ ]:
# Turn on the two full runs only when you are ready to spend the quota.
RUN_FULL_WEEK3 = False       # the agent on all 100 questions (results/week3/agent.jsonl)
RUN_NO_CRITIC = False        # the same run without the critic, to measure what the critic adds

RUN_SAMPLES = True           # small live runs on SAMPLE_IDS
RUN_API = True               # start the Week 4 API and test it
RUN_UI = True                # start the Streamlit UI (needs RUN_API)
API_PORT, UI_PORT = 8100, 8501

# Mix of easy, reworded, boundary, hard-verdict and must-refuse questions (same as the Week 1 and 2 notebook).
SAMPLE_IDS = ["Q001", "Q003", "Q071", "Q075", "Q026", "Q062", "Q080", "Q087", "Q094", "Q100"]

BRANCH = "aakash_r_changes"
DRIVE_FOLDER = "medevidence-copilot-colab"
BUNDLE_NAME = "medevidence-copilot-week3-4.bundle"

## Setup

### 1. Connect Google Drive and define helpers
Drive holds the code, keys and certificates, and keeps the saved model answers, the search index and the final results.

In [ ]:
from collections import deque
import hashlib, json, os, shutil, sqlite3, subprocess, sys, tempfile, textwrap, threading, time, zipfile
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
from IPython.display import Markdown, display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    WORK = Path("/content")
else:  # running outside Colab, for example to test the notebook
    DRIVE = Path(os.environ["MEDEV_DRIVE"])
    WORK = Path(os.environ["MEDEV_WORK"])

REPO = WORK / "medevidence-copilot"
REFERENCE = WORK / "committed_results"
for d in ("certs", "cache", "runs", "index"):
    (DRIVE / d).mkdir(parents=True, exist_ok=True)


def secret(name):
    path = DRIVE / "secrets" / "medevidence.env"
    if not path.exists():
        raise RuntimeError(f"Missing {path}")
    for line in path.read_text().splitlines():
        key, sep, value = line.strip().partition("=")
        if sep and key == name and value:
            return value.strip()
    raise RuntimeError(f"{name} is not set in secrets/medevidence.env")


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def save_cache():
    # Copies the saved model answers to Drive, safely, even while a run is writing them.
    for src in (REPO / ".cache").glob("*.sqlite"):
        if src.name == "sessions.sqlite":
            continue
        with tempfile.TemporaryDirectory() as tmp:
            copy = Path(tmp) / src.name
            a, b = sqlite3.connect(src), sqlite3.connect(copy)
            with b:
                a.backup(b)
            a.close(), b.close()
            part = DRIVE / "cache" / (src.name + ".part")
            shutil.copyfile(copy, part)
            os.replace(part, DRIVE / "cache" / src.name)


def run(script, *args, every=120, quiet=False):
    # Runs a project script, shows its output (or only its end if it fails) and saves the cache every 2 minutes.
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "30"}
    proc = subprocess.Popen([sys.executable, script, *args], cwd=REPO, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    done = threading.Event()

    def saver():
        while not done.wait(every):
            save_cache()

    threading.Thread(target=saver, daemon=True).start()
    tail = deque(maxlen=30)
    for line in proc.stdout:
        if line.strip():
            tail.append(line.rstrip())
            if not quiet:
                print(line.rstrip(), flush=True)
    code_ = proc.wait()
    done.set()
    save_cache()
    if code_:
        raise RuntimeError(f"{script} failed with exit code {code_}\n" + "\n".join(tail))
    if quiet:
        print("Finished", script, *args)


def pct(x):
    return "n/a" if x is None else f"{x * 100:.1f}%"


def wrap(s, indent="    "):
    return textwrap.fill(str(s), width=100, initial_indent=indent, subsequent_indent=indent)


print("Drive folder:", DRIVE)

### 2. Get the code

In [ ]:
BUNDLE = DRIVE / "code" / BUNDLE_NAME
if not REPO.exists():
    if not BUNDLE.exists():
        raise RuntimeError(f"Missing {BUNDLE}")
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, str(BUNDLE), str(REPO)], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO), "log", "-1", "--format=%h  %s"], text=True).strip()
print("Code version:", commit)
if not (REPO / "src/medevidence/agent").exists():
    raise RuntimeError("This code version has no Week 3 agent: the bundle in Drive is too old")

# Keep an untouched copy of the committed results, to compare with at the end.
if not REFERENCE.exists():
    shutil.copytree(REPO / "results", REFERENCE)

### 3. Install packages
Colab already has torch with GPU support, so it is not reinstalled.

In [ ]:
packages = ["httpx==0.28.1", "rank-bm25==0.2.2", "python-dotenv", "tqdm", "pandas", "pyarrow", "numpy",
            "transformers", "sentence-transformers", "opentelemetry-api", "opentelemetry-sdk",
            "faiss-cpu", "rouge-score", "streamlit", "requests", "pytest"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import torch
GPU = torch.cuda.is_available()
os.environ["RERANK_DEVICE"] = "cuda" if GPU else "cpu"
print("Packages ready. Device:", "GPU" if GPU else "no GPU (the reranker will be slow)")

### 4. Connect to the course endpoints
Keys are read from the private `secrets` folder in Drive and are never printed.

In [ ]:
CERTS = {"CA_CERT_PATH": "vm04-ca.crt", "EMBEDDING_CA_CERT_PATH": "vm02-ca.crt"}

missing = [name for name in CERTS.values() if not (DRIVE / "certs" / name).exists()]
if missing:
    raise RuntimeError(f"Missing certificates in Drive: {missing}")

for name in ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL", "EMBEDDING_API_KEY", "EMBEDDING_BASE_URL", "EMBEDDING_MODEL"):
    os.environ[name] = secret(name)
for variable, name in CERTS.items():
    os.environ[variable] = str(DRIVE / "certs" / name)

HF_DIR = WORK / "pubmedqa_parquet"
os.environ["PUBMEDQA_HF_DIR"] = str(HF_DIR)
os.environ["MEDEVIDENCE_TRACE_FILE"] = str(REPO / "results/traces/spans.jsonl")
print("Credentials loaded. Model:", os.environ["LLM_MODEL"])

### 5. Get the data
Same files as Weeks 1 and 2: the expert-labelled PubMedQA files from `capstone.zip`, and two distractor files from Hugging Face, pinned to one exact version and checked by hash.

In [ ]:
RAW = REPO / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(REPO / "capstone.zip") as z:
    for name in ("ori_pqal.json", "test_ground_truth.json"):
        (RAW / name).write_bytes(z.read(f"capstone/data/raw/{name}"))

HF_REVISION = "9001f2853fb87cab8d220904e0de81ac6973b318"
PARQUET = {"pqa_artificial": "cdee2d3c01369f47d2ea7b89cd32ee64b142cc0debcb863bf39b3a81a3cece05",
           "pqa_unlabeled": "2f390ea5e6e3dc53b2736a52be2f594e05f5d8a599c3de754ad72f1068251e6e"}
HF_DIR.mkdir(exist_ok=True)
for name, expected in PARQUET.items():
    path = HF_DIR / f"{name}.parquet"
    if not path.exists() or sha256(path) != expected:
        print("Downloading", name)
        urlretrieve(f"https://huggingface.co/datasets/qiaojin/PubMedQA/resolve/{HF_REVISION}/{name}/train-00000-of-00001.parquet", path)
    if sha256(path) != expected:
        raise RuntimeError(f"{name} does not match the expected file")

EVAL_SHA = "1a5befac61c86ce450dfc84e32f0940789d171f1b622f0a2a89863eb23ba6e56"
CORPUS_SHA = "da23af8479e15b7f3462aa40a65b91a53597ad8e7aeb11e1ad44b7a85e9e99cb"
run("scripts/build_eval_set.py", quiet=True)
run("scripts/build_corpus.py", quiet=True)
if sha256(REPO / "eval/eval_set.jsonl") != EVAL_SHA:
    raise RuntimeError("The rebuilt eval set differs from the frozen one")
if sha256(REPO / "data/corpus.jsonl") != CORPUS_SHA:
    raise RuntimeError("The corpus differs from the one used for the committed results")
print("Eval set and corpus match the frozen versions.")

### 6. Restore saved answers, build the index and the FAISS store
Saved model answers and embeddings come back from Drive, so nothing is paid for twice. The agent's search tool needs the embedded chunks (`build_index.py`) and the FAISS store (`build_faiss.py`). The index is copied to Drive, so the next run skips this step.

In [ ]:
(REPO / ".cache").mkdir(exist_ok=True)
for saved in (DRIVE / "cache").glob("*.sqlite"):
    target = REPO / ".cache" / saved.name
    if not target.exists():
        shutil.copyfile(saved, target)
        print("Restored", saved.name)

INDEX = REPO / "data/index"
if not (INDEX / "q_prefixed.npy").exists() and (DRIVE / "index" / "q_prefixed.npy").exists():
    shutil.copytree(DRIVE / "index", INDEX, dirs_exist_ok=True)
    print("Restored the index from Drive")
if not (INDEX / "q_prefixed.npy").exists():
    run("scripts/build_index.py")
if not (INDEX / "faiss_abstract_flat.index").exists():
    run("scripts/build_faiss.py")
shutil.copytree(INDEX, DRIVE / "index", dirs_exist_ok=True)

display(pd.DataFrame([{"chunking": n, "chunks": len(json.load(open(INDEX / f"chunks_{n}.json")))}
                      for n in ("abstract", "section", "window")]))

### 7. Load the project code and test the connection

In [ ]:
sys.path[:0] = [str(REPO / "src"), str(REPO / "scripts")]
from medevidence import data, judge, metrics
from medevidence.llm import LLM
from medevidence.agent import Agent, build_agent, safety

try:
    llm = LLM()
    reply = llm.chat([{"role": "user", "content": f"Reply with the word OK. Test {time.time()}"}], max_tokens=5)
except Exception as e:
    raise RuntimeError("Could not reach the course endpoints. Check secrets/medevidence.env and the certificates. "
                       "If they are right, the endpoints may not accept connections from Colab, "
                       f"so run the project on your own machine instead.\nDetails: {type(e).__name__}") from None
print(f"Chat endpoint OK ({reply.latency:.1f}s).")

ev = data.load_eval()
by_id = {item["id"]: item for item in ev}
agent = build_agent()          # with sessions, as the API uses it
corpus = agent.tools.abs
print("Agent ready:", len(corpus), "abstracts, FAISS:", agent.tools.index.faiss is not None,
      "| reranker on", os.environ["RERANK_DEVICE"])

---
# Week 3: the agent

```
question ──► safety gate ──refuse──► fixed refusal template
                 │ (rules + LLM classifier)
                 ▼
             planner ──► research question + up to 3 search queries
                 ▼
             executor ──► tools: search_abstracts, get_abstract, verify_quote, conclusion_sentences
                 ▼                  (re-plans once if nothing relevant is found)
             draft: verdict, explanation, verbatim quote, PMID
                 ▼
             critic ──► code checks + LLM check; sends back up to 2 times, then downgrades to "maybe"
```

## 1. Unit tests
27 tests with a scripted fake model: no endpoint, no quota.

In [ ]:
run("-m", "pytest", "-q", "tests")

## 2. The safety gate on its own
The rules run before anything else. An emergency hit refuses at once, without a model call. Research questions that only *mention* dosing or chest pain must pass.

In [ ]:
examples = [
    "Does CBT reduce suicidal ideation in adolescents?",
    "Do higher loading doses of vancomycin improve target attainment?",
    "Can we predict outcome in chest pain patients?",
    "My dad just collapsed and is not breathing",
    "I have crushing chest pain right now",
    "How many mg of ibuprofen should I give my son?",
    "My PSA went from 4 to 7, does that mean I have cancer?",
]
rows = []
for q in examples:
    cat, rule = safety.rule_check(q)
    rows.append({"question": q, "rule category": cat or "(passes)", "rule": rule or ""})
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(rows))
print("Emergency refusal text:\n" + wrap(safety.refusal("emergency")))

## 3. Follow one question through the agent
The reworded hip-fracture question from Week 2. Every step is a span in the OpenTelemetry trace.

In [ ]:
def show(r):
    print(f"Status: {r['status']}   Verdict: {r.get('verdict')}   PMID: {r.get('citation_pmid')}   "
          f"quote verified: {r.get('quote_verified')}   {r['latency_s']:.1f}s")
    if r.get("message"):
        print("Message:"); print(wrap(r["message"]))
    if r.get("plan"):
        print("Planner's research question:"); print(wrap(r["plan"].get("research_question")))
        print("Search queries:", r["plan"].get("queries"))
    if r.get("retrieved"):
        print("Abstracts read:", [d["pmid"] for d in r["retrieved"]])
    if r.get("explanation"):
        print("Explanation:"); print(wrap(r["explanation"]))
        print("Quote:"); print(wrap(r.get("evidence_quote")))
    if r.get("critic"):
        c = r["critic"]
        print(f"Critic: approved {c['approved']}, revisions {c['revisions']}, downgraded {c['downgraded']}, "
              f"draft verdicts {c.get('draft_verdicts')}")
    u = r["usage"]
    print(f"Usage: {u['llm_calls']} model calls, {u['tool_calls']} tool calls, {u['tokens']} tokens, {u['steps']} steps")


item = by_id["Q003"]
print("Question:", item["query"])
print("Correct answer:", item["gold_label"], "| correct PMID:", item["gold_pmid"], "\n")
demo = agent.ask(item["query"], session_id="colab-demo")
show(demo)
display(pd.DataFrame(demo["trace"]))
save_cache()

### The same request as an OpenTelemetry trace
Spans are written to `results/traces/spans.jsonl`. Indentation shows which step ran inside which.

In [ ]:
def trace_tree(trace_id):
    path = REPO / "results/traces/spans.jsonl"
    spans = [json.loads(l) for l in open(path)] if path.exists() else []
    spans = [s for s in spans if s["trace_id"] == trace_id]
    if not spans:
        print("No spans found (is opentelemetry installed?)")
        return
    kids = {}
    for s in spans:
        kids.setdefault(s["parent_id"], []).append(s)
    ids = {s["span_id"] for s in spans}

    def walk(s, depth):
        a = s["attributes"]
        extra = ", ".join(f"{k}={a[k]}" for k in ("prompt_tokens", "completion_tokens", "cached", "category", "n") if k in a)
        print(f"{'  ' * depth}{s['name']:<28} {s['duration_ms']:>8.1f} ms  {extra}")
        for c in sorted(kids.get(s["span_id"], []), key=lambda x: x["start_ns"]):
            walk(c, depth + 1)

    for root in sorted([s for s in spans if s["parent_id"] not in ids], key=lambda x: x["start_ns"]):
        walk(root, 0)


trace_tree(demo["trace_id"])

## 4. A follow-up in the same session
Sessions are stored in SQLite, so the planner sees the earlier question and resolves "and in ...?" on its own.

In [ ]:
follow = agent.ask("Is that also true after hip replacement surgery?", session_id="colab-demo")
show(follow)
save_cache()

## 5. The agent on the sample
Scored with the same checks C1 to C6 as Weeks 1 and 2, plus the failure class used in the Week 3 report.

In [ ]:
import run_week3
from medevidence.rouge import rouge_l


def score(item, r):
    o, err = run_week3.pred_obj(r)
    s = metrics.score_item(item, o, err, corpus, citation_pmid=r.get("citation_pmid"), has_citation=True)
    if item["expected_behavior"] == "answer":
        if r["status"] == "answered" and r.get("citation_pmid") in corpus:
            j = judge.judge(llm, item, corpus[r["citation_pmid"]], r.get("explanation"))
            s["c4_faithful"], s["c4_unsupported"] = judge.faithful(j), j["unsupported"]
        else:
            s["c4_faithful"] = False
        s["rougeL"] = round(rouge_l(r.get("explanation") if r["status"] == "answered" else None, item["gold_rationale"])["f"], 3)
    metrics.finish_strict(s)
    s["failure"] = run_week3.taxonomy(item, r, s, corpus)
    return s


if RUN_SAMPLES:
    eval_agent = Agent(llm, agent.tools)   # no session history, as in the evaluation; shares the loaded index
    table = []
    for qid in SAMPLE_IDS:
        item = by_id[qid]
        r = eval_agent.ask(item["query"])
        s = score(item, r)
        said = (f"refused: {r['refusal_category']}" if r["status"] == "refused"
                else r.get("verdict") if r["status"] == "answered" else r["status"])
        table.append({"id": qid, "type": item["query_type"],
                      "correct": item["gold_label"] or f"refuse: {item['refusal_category']}",
                      "agent": said, "pass": bool(s.get("strict") if item["expected_behavior"] == "answer" else s.get("correct")),
                      "failure": s["failure"] or "", "critic revisions": (r.get("critic") or {}).get("revisions"),
                      "calls (model/tool)": f"{r['usage']['llm_calls']}/{r['usage']['tool_calls']}",
                      "seconds": round(r["latency_s"], 1), "ROUGE-L": s.get("rougeL")})
        print("finished", qid, flush=True)
    display(pd.DataFrame(table))
    save_cache()
else:
    print("Skipped: RUN_SAMPLES is False")

## 6. Full Week 3 run and the critic ablation
Each runs only when its setting is `True`. Otherwise the next cells use the committed results, so the rest of the notebook still works.

In [ ]:
if RUN_FULL_WEEK3:
    run("scripts/run_week3.py", "--name", "agent")
else:
    print("Skipped the full run: using the committed Week 3 results.")
if RUN_NO_CRITIC:
    run("scripts/run_week3.py", "--name", "agent_nocritic", "--no-critic")
else:
    print("Skipped the critic ablation.")

## 7. Week 3 results
**Strict** means the answer passes every check, C1 to C6, over the 78 answerable questions.

In [ ]:
now = json.load(open(REPO / "results/week3/summary.json"))
committed = json.load(open(REFERENCE / "week3/summary.json"))
METRICS = [("strict_all", "strict (C1-C6)"), ("verdict_acc_all", "verdict (C1)"), ("macro_f1", "macro-F1"),
           ("citation_acc", "citation (C2)"), ("grounded_acc", "quote exact (C3)"), ("faithful_acc", "faithful (C4)"),
           ("format_all", "format (C5)"), ("hallucinated_pmids", "hallucinated PMIDs"),
           ("refusal_recall", "refused"), ("refusal_correct", "refusal fully right"),
           ("emergency_refused", "emergencies refused"), ("over_refusal_rate", "wrongly refused"),
           ("gold_in_context", "right abstract in context")]
rows = []
for key, label in METRICS:
    row = {"metric": label}
    for name, s in [*now.items(), ("agent (committed)", committed.get("agent", {}))]:
        v = s.get(key)
        row[name] = round(v, 3) if key == "macro_f1" and v is not None else pct(v) if isinstance(v, float) else v
    rows.append(row)
display(pd.DataFrame(rows))
print("Source:", "this run" if RUN_FULL_WEEK3 else "committed results")

base = now["agent"].get("vs_baselines", {})
display(Markdown("**Paired bootstrap, 10k resamples: agent minus baseline**"))
display(pd.DataFrame([{"comparison": k, "agent": pct(v["agent"]), "baseline": pct(v["baseline"]),
                       "difference": f"{v['diff'] * 100:+.1f} pts",
                       "95% CI": f"[{v['ci95'][0] * 100:+.1f}, {v['ci95'][1] * 100:+.1f}]",
                       "proven improvement": v["improvement"]} for k, v in base.items()]))

### Observability, the critic and the failures

In [ ]:
s = now["agent"]
display(pd.DataFrame([
    {"measure": "task completion rate", "value": pct(s["task_completion_rate"])},
    {"measure": "model calls per question", "value": round(s["avg_llm_calls"], 2)},
    {"measure": "tool calls per question", "value": round(s["avg_tool_calls"], 2)},
    {"measure": "tokens per question", "value": round(s["avg_tokens"])},
    {"measure": "estimated cost per question (USD)", "value": f"{s['avg_cost_usd_est']:.5f}"},
    {"measure": "latency median / p90 (s)", "value": f"{s['latency_median_s']:.1f} / {s['latency_p90_s']:.1f}"},
    {"measure": "JSON repairs / tool errors", "value": f"{s['json_repairs']} / {s['tool_errors']}"},
]))
display(Markdown("**What the critic did** (answerable questions)"))
display(pd.DataFrame([s["critic"]]))
if "agent_nocritic" in now:
    n = now["agent_nocritic"]
    display(Markdown("**Critic ablation**"))
    display(pd.DataFrame([{"run": k, "strict": pct(now[k]["strict_all"]), "verdict": pct(now[k]["verdict_acc_all"]),
                           "tokens / question": round(now[k]["avg_tokens"])} for k in ("agent", "agent_nocritic")]))

display(Markdown("**Why questions fail**"))
display(pd.DataFrame(sorted(s["failure_taxonomy"].items(), key=lambda x: -x[1]), columns=["failure", "questions"]))

rows = [json.loads(l) for l in open(REPO / "results/week3/agent.jsonl")]
wrong = pd.crosstab(pd.Series([r["gold"] for r in rows if r.get("failure") == "reasoning_error"], name="correct"),
                    pd.Series([r["verdict"] for r in rows if r.get("failure") == "reasoning_error"], name="agent"))
display(Markdown("**Reasoning errors: correct verdict down the side, the agent's across the top**"))
display(wrong)

### One example of each failure class
Read from the saved results, no model calls.

In [ ]:
seen = set()
for r in rows:
    f = r.get("failure")
    if not f or f in seen:
        continue
    seen.add(f)
    item = by_id[r["id"]]
    pred = r.get("pred") or {}
    print("=" * 100)
    print(f"{f}  ·  {r['id']} ({item['query_type']})")
    print("=" * 100)
    print(wrap(item["query"]))
    print("Correct:", item["gold_label"] or f"REFUSE as {item['refusal_category']}")
    print("Agent:  ", r["verdict"] if r["status"] == "answered" else f"{r['status']} {pred.get('refusal_category') or ''}")
    if pred.get("explanation"):
        print("Explanation:"); print(wrap(pred["explanation"]))
    feedback = (r.get("critic_notes") or [{}])[-1].get("feedback")
    if feedback:
        print("Critic's last note:"); print(wrap("; ".join(feedback) if isinstance(feedback, list) else feedback))
    print()

## 8. ROUGE-L and the FAISS check
ROUGE-L compares each explanation with the authors' conclusion (used only for scoring). It is recomputed from the saved rows of every run, so it needs no model calls.

In [ ]:
run("scripts/eval_rouge.py", quiet=True)
rouge = json.load(open(REPO / "results/rouge_l.json"))["summary"]
keep = ["week1/closed__fewshot", "week1/bm25__fewshot", "week2/rag__abstract__bm25_1", "week2/rag__abstract__rerank_3",
        *[k for k in rouge if k.startswith("week3/")]]
display(pd.DataFrame([{"run": k, "all 78": v["rougeL_all"], "answered": v["rougeL_answered"],
                       "right verdict": v["verdict_right"], "wrong verdict": v["verdict_wrong"]}
                      for k, v in rouge.items() if k in keep]))

faiss_check = json.load(open(REPO / "results/faiss_check.json"))
display(Markdown("**FAISS against the numpy scan, 100 eval queries**"))
display(pd.DataFrame([{"store": k, "top-10 identical": pct(v["top10_identical_to_numpy"]), "R@1": pct(v["R@1"]),
                       "R@5": pct(v["R@5"]), "ms / query": v["ms_per_query"]} for k, v in faiss_check.items()]))

---
# Week 4: ship it

## 9. Start the API
`app/server.py` runs in the background on port 8100, with the agent behind it. It loads its own copy of the index and the reranker, so the first start takes a minute or two.

In [ ]:
import requests

API = f"http://127.0.0.1:{API_PORT}"
api_proc = None
if RUN_API:
    api_env = {**os.environ, "HOST": "127.0.0.1", "PORT": str(API_PORT), "MEDEVIDENCE_MODE": "agent",
               "RATE_LIMIT_PER_MIN": "10", "RATE_LIMIT_BURST": "5", "CACHE_TTL_S": "3600"}
    api_log = open(WORK / "api.log", "w")
    api_proc = subprocess.Popen([sys.executable, "app/server.py"], cwd=REPO, env=api_env,
                                stdout=api_log, stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            health = requests.get(API + "/api/health", timeout=2).json()
            break
        except requests.RequestException:
            if api_proc.poll() is not None:
                raise RuntimeError("The API stopped:\n" + (WORK / "api.log").read_text()[-2000:])
            time.sleep(3)
    else:
        raise RuntimeError("The API did not start in 6 minutes:\n" + (WORK / "api.log").read_text()[-2000:])
    print(json.dumps(health, indent=1))
else:
    print("Skipped: RUN_API is False")

## 10. Caching, sessions and rate limiting
- The same question twice: the second answer comes from the answer cache, with no model call.
- A follow-up with a `session_id` skips the cache, because its answer depends on the history.
- The limit is a token bucket per client: 5 requests at once, then 10 a minute. Rapid repeats of a cached question are cheap, so they show the HTTP 429 without spending quota.

In [ ]:
def ask(question, session_id=None):
    t0 = time.time()
    resp = requests.post(API + "/api/ask", json={"question": question, "session_id": session_id}, timeout=300)
    body = resp.json()
    return resp.status_code, body, time.time() - t0


if RUN_API:
    q = by_id["Q001"]["query"]
    rows = []
    for label, sid in (("first ask", None), ("same question again", None)):
        code_, body, secs = ask(q, sid)
        rows.append({"request": label, "HTTP": code_, "status": body.get("status"), "verdict": body.get("verdict"),
                     "cached": body.get("cached"), "seconds": round(secs, 2)})
    code_, body, secs = ask(q, "api-demo")
    rows.append({"request": "new session (cached answer, saved to its history)", "HTTP": code_, "status": body.get("status"), "verdict": body.get("verdict"),
                 "cached": body.get("cached"), "seconds": round(secs, 2)})
    code_, body, secs = ask("Does the same hold for amateur athletes?", "api-demo")
    rows.append({"request": "follow-up in that session", "HTTP": code_, "status": body.get("status"),
                 "verdict": body.get("verdict"), "cached": body.get("cached"), "seconds": round(secs, 2)})
    display(pd.DataFrame(rows))
    print("Question:", q)

    burst = []
    for i in range(8):
        resp = requests.post(API + "/api/ask", json={"question": q}, timeout=60)
        burst.append({"request": i + 1, "HTTP": resp.status_code, "Retry-After": resp.headers.get("Retry-After"),
                      "cached": resp.json().get("cached")})
    display(Markdown("**8 requests in a row from one client**"))
    display(pd.DataFrame(burst))

    fb = requests.post(API + "/api/feedback", json={"question": q, "rating": "up", "note": "Colab test"}).json()
    print("Feedback endpoint:", fb)
    print("Health:", requests.get(API + "/api/health").json().get("cache"))
    save_cache()

## 11. The Streamlit UI
Starts `app/streamlit_app.py` against the API and opens it through Colab's port proxy. If the frame stays blank, use the link printed below it.

In [ ]:
ui_proc = None
if RUN_API and RUN_UI:
    ui_log = open(WORK / "ui.log", "w")
    ui_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app/streamlit_app.py",
                                f"--server.port={UI_PORT}", "--server.headless=true", "--browser.gatherUsageStats=false",
                                "--server.enableCORS=false", "--server.enableXsrfProtection=false"],
                               cwd=REPO, env={**os.environ, "API_URL": API}, stdout=ui_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            requests.get(f"http://127.0.0.1:{UI_PORT}/_stcore/health", timeout=2)
            break
        except requests.RequestException:
            time.sleep(2)
    if IN_COLAB:
        from google.colab import output
        print("Open in a new tab:", output.eval_js(f"google.colab.kernel.proxyPort({UI_PORT})"))
        output.serve_kernel_port_as_iframe(UI_PORT, height=900)
    else:
        print(f"UI running at http://127.0.0.1:{UI_PORT}")
else:
    print("Skipped.")

## 12. Docker
Colab cannot run Docker, so this only shows the setup. On your own machine, with `.env`, `certs/` and `data/` in the repo folder:

```bash
docker compose up --build    # UI: http://127.0.0.1:8501, API: http://127.0.0.1:8100
```

Keys, certificates and data are mounted at run time and never copied into the image.

In [ ]:
print((REPO / "docker-compose.yml").read_text())
print((REPO / "Dockerfile").read_text())

---
# Finish

## 13. Save everything and stop the servers
Results, traces and regenerated reports go to a dated folder in Drive. The list shows which tracked files differ from the commit. Nothing is committed or pushed.

In [ ]:
for proc in (globals().get("ui_proc"), globals().get("api_proc")):
    if proc is not None and proc.poll() is None:
        proc.terminate()

save_cache()
stamp = time.strftime("%Y-%m-%d_%H%M%S")
target = DRIVE / "runs" / f"week3-4_{stamp}"
shutil.copytree(REPO / "results", target / "results")
shutil.copytree(REPO / "docs", target / "docs", ignore=shutil.ignore_patterns("*.pdf"))
for log in ("api.log", "ui.log"):
    if (WORK / log).exists():
        shutil.copyfile(WORK / log, target / log)

changed = subprocess.check_output(["git", "-C", str(REPO), "status", "--short", "--", "eval", "results", "docs"], text=True).strip()
print("Saved to:", target)
print("\nFiles that differ from the commit:")
print(changed or "none: everything reproduced exactly")